In [ ]:
from pyspark.sql import SparkSession
from google.cloud import storage
import json

In [ ]:
'''
!mkdir -p "$HOME/spark-jars"

!curl -fL \
  "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar" \
  -o "$HOME/spark-jars/gcs-connector-hadoop3-latest.jar"
'''

In [ ]:
#!ls -lh "$HOME/spark-jars/gcs-connector-hadoop3-latest.jar"

In [ ]:
import os
from pyspark.sql import SparkSession

gcs_connector_jar = os.path.expanduser(
    "~/spark-jars/gcs-connector-hadoop3-latest.jar"
)

if not os.path.isfile(gcs_connector_jar):
    raise FileNotFoundError(
        f"Nie znaleziono konektora: {gcs_connector_jar}"
    )

spark = (
    SparkSession.builder
    .appName("development_for_silver_layer")
    .config(
        "spark.jars",
        gcs_connector_jar
    )
    .config(
        "spark.hadoop.fs.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem"
    )
    .config(
        "spark.hadoop.fs.AbstractFileSystem.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS"
    )
    .getOrCreate()
)

In [ ]:
spark = SparkSession \
        .builder\
        .appName("development_for_silver_layer") \
        .getOrCreate()

In [ ]:
spark.version

In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 50)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

In [ ]:
spark.conf.get(
    "spark.sql.adaptive.advisoryPartitionSizeInBytes"
)

In [ ]:
storage_client = storage.Client()
bucket = storage_client.get_bucket("project-dev-storage")

In [ ]:
fil = json.loads(data_string)

#df = spark.createDataFrame(fil)
#df

In [ ]:
print(
    spark.sparkContext
    ._jsc
    .hadoopConfiguration()
    .get("fs.gs.impl")
)

In [ ]:
df_metadata = (
    spark.read
    .format("json")
    #.option("multiline", "true")
    .load("gs://project-dev-storage/company_forms/sec_metadata_folder/2026-08-06/")
)

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df_metadata.select(
    "sic",
    "cik",
    "name",
   # "entityType",
    "sicDescription",
    "fiscalYearEnd",
    F.col("tickers")[0].alias("tickers"),
    F.col("exchanges")[0].alias("exchanges")
    ).show()

In [ ]:
df_test =  df_metadata.select(
        "filings.recent.accessionNumber",
        "filings.recent.filingDate",
        "filings.recent.reportDate",
        "filings.recent.form",
        "filings.recent.primaryDocument",
        "filings.recent.isXBRL",
        "filings.recent.isInlineXBRL"
)

df_test.show()

In [ ]:
df_test

In [ ]:
df_test2 = df_test.select(F.explode(F.arrays_zip(df_test.accessionNumber, df_test.filingDate, df_test.reportDate, df_test.form, df_test.primaryDocument, df_test.isXBRL, df_test.isInlineXBRL)).alias("x")).select(F.col("x.accessionNumber"), F.col("x.filingDate"), F.col("x.reportDate"),
                                                                                                                                                   F.col("x.form"), F.col("x.primaryDocument"), F.col("x.isXBRL"), F.col("x.isInlineXBRL"))
df_test2

In [ ]:
aqe_enabled = spark.conf.get("spark.sql.adaptive.enabled")

print(aqe_enabled)

In [ ]:
#df_test.write.format("delta").mode("overwrite").save("gs://project-dev-storage/company_forms/sec_metadata_folder/2026-08-06/test")
df_test2.hint("REBALANCE").write.mode("overwrite").parquet("gs://project-dev-storage/company_forms/sec_metadata_folder/2026-08-06/test")

In [ ]:
################################################################# XBRL ##################################################################

In [ ]:
# 3 XBRL
df_content = (
    spark.read
    .format("json")
    .load("gs://project-dev-storage/company_forms/sec_facts_folder/2026-08-06/")
)

In [ ]:
#df_temp = spark.read.json("gs://project-dev-storage/company_forms/sec_facts_folder/2026-08-06/0001045810_fact_file_2026-08-06 20:25:53.999746+02:00")
#df_temp.show()

In [ ]:
client = storage.Client()
blob = client.bucket("project-dev-storage").blob("company_forms/sec_facts_folder/2026-08-06/0000320193_fact_file_2026-08-06 20:25:53.999746+02:00")
json_content = blob.download_as_text(encoding="utf-8")

In [ ]:
json_file = json.loads(json_content)
type(json_file)

In [ ]:
json_file["facts"]["us-gaap"]["AcceleratedShareRepurchaseProgramAdjustment"].keys()

In [ ]:
#json_file["filings"]["recent"]["accessionNumber"]#.keys()

In [ ]:
'''
df_content.select(
        "cik",
        "entityName",
        "f
).show()
'''

In [ ]:
#df_content.select(F.explode("facts"))
df_content

In [ ]:
# ZAMIAST AcceleratedShareRepurchaseProgramAdjustment to trzeba by liste po czym ma isc pętla
# "facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.units.val"
    # DO WERYFIKACJI CZY zawsze facts.us-gaap czy cos innego ?
df_content2 = df_content.select("cik","entityname", "facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.label", F.explode(F.arrays_zip("facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.units.usd.val", "facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.units.usd.start"))
    .alias("x")).select(F.col("cik"), F.col("entityname"), F.col("label"), F.col("x.val"), F.col("x.start"))
df_content2.show(truncate = False)

In [ ]:
# pobranie nazw kolumn ze schematu
us_gaap_fields = df_content.select("facts.us-gaap.*").columns
us_gaap_fields = us_gaap_fields[:5]

In [ ]:
us_gaap_fields

In [ ]:
for i in us_gaap_fields:
    print(type(i))

In [ ]:
#df_content_new.select("facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.units.usd.val")

In [ ]:
df_content_new = df_content.limit(1)
df_content_new

In [ ]:
df_content_new.show()

In [ ]:
#df_content_new.schema["facts"].dataType["us-gaap"] # -> StrunctField tych fieldow ktore zawieraja sie w us-gaap

In [ ]:
# LABELS WITH USD ONLY

us_gaap_schema = (
                df_content_new.schema["facts"].dataType["us-gaap"].dataType

)

type(us_gaap_schema)
#us_gaap_schema

In [ ]:
#us_gaap_schema

In [ ]:
us_gaap_fields = df_content.select("facts.us-gaap.*").columns
#us_gaap_fields
#us_gaap_fields = us_gaap_fields[:5]

In [ ]:
##################################################### TESTY ################################

In [ ]:
from pyspark.sql.types import StructType

In [ ]:
#us_gaap_schema

In [ ]:
for i in us_gaap_fields:
    fact_schema = us_gaap_schema[i].dataType
    print(fact_schema)
    #if "units" in fact_schema.fieldNames():
    #    units_schema = fact_schema["units"].dataType
    #    print(units_schema)
    break

In [ ]:
from pyspark.sql.types import ArrayType

In [ ]:
usd_labels = []
for i in us_gaap_fields:
    fact_schema = us_gaap_schema[i].dataType
    if "units" in fact_schema.fieldNames():
        units_schema = fact_schema["units"].dataType
        #print(units_schema)
        if isinstance(units_schema, StructType):
            if "USD" in units_schema.fieldNames():
               units_schema2 = units_schema["USD"].dataType
               #rint(units_schema2)
               if isinstance(units_schema2, ArrayType):
                   usd_element_schema = units_schema2.elementType
                   #print(usd_element_schema)
                   #print(usd_element_schema.fieldNames())
                   if "start" in usd_element_schema.fieldNames():
                       usd_labels.append(i)     
               



''' KOPIA

usd_labels = []
for i in us_gaap_fields:
    fact_schema = us_gaap_schema[i].dataType
    if "units" in fact_schema.fieldNames():
        units_schema = fact_schema["units"].dataType
        #print(units_schema)
        if isinstance(units_schema, StructType):
            if "USD" in units_schema.fieldNames():
               usd_labels.append(i) 

''' 

In [ ]:
len(usd_labels)

In [ ]:
usd_labels

In [ ]:
us_gaap_fields = us_gaap_fields[:5]
us_gaap_fields

In [ ]:
usd_labels = usd_labels[:5]
usd_labels

In [ ]:
df_content_new.show()

In [ ]:
for i in usd_labels:
    print(i)

In [ ]:
df_content2 = df_content_new.select("cik","entityname", 
                                                              
                       F.explode(F.array(*[F.struct
                                  (
                                    F.col(f"facts.us-gaap.{i}.label").alias("label"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.start").alias("start")
                                  )
                                for i in usd_labels])
                            ).alias("fact")
                
                    ).select("cik","entityname","fact.label","fact.start")
                                          
                           
                              
df_content2.show()

In [ ]:
df_content2.limit(20).toPandas()

In [ ]:
df_content2 = df_content_new.select("cik","entityname", 
                                        
                              
                        F.explode(F.array(*[F.struct
                                  (
                                    F.col(f"facts.us-gaap.{i}.label").alias("label"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.val").alias("val")
                                  )
                                for i in usd_labels]
                                )
                            ).alias("fact")      
                        ).select("cik", "entityname","fact.label","fact.val")               
                                      
                           
                              
df_content2.show()

In [ ]:

df_content2 = df_content_new.select("cik","entityname", F.explode(F.array(*[f"facts.us-gaap.{i}.label" for i in usd_labels])).alias("label"),
                                    *[f"facts.us-gaap.{i}.units.USD.val" for i in usd_labels]
                                    #*[f"facts.us-gaap.{i}.units.usd" for i in us_gaap_fields]
                                    #F.arrays_zip(*[f"facts.us-gaap.{i}.units.usd.val" for i in us_gaap_fields])
                              )
                              
df_content2.show()


In [ ]:
#df_content2.columns

In [ ]:
'''
                 KOPIA
df_content2 = df_content_new.select("cik","entityname", F.explode(F.array(*[f"facts.us-gaap.{i}.label" for i in us_gaap_fields])).alias("label"),
                                    *[f"facts.us-gaap.{i}.units.USD.val" for i in usd_labels]
                                    #*[f"facts.us-gaap.{i}.units.usd" for i in us_gaap_fields]
                                    #F.arrays_zip(*[f"facts.us-gaap.{i}.units.usd.val" for i in us_gaap_fields])
                              )
                              
df_content2.show()
'''